In [ ]:
"""
Prompt Guardrail for Ollama
============================
Checks a user's prompt for prompt-injection attempts / unsafe instructions
BEFORE sending it to a local LLM running in Ollama.

Requirements:
    pip install ollama

Usage:
    python guardrail_ollama.py
    (make sure `ollama serve` is running and you have pulled a model,
     e.g. `ollama pull llama3`)
"""

import re
import sys

try:
    import ollama
except ImportError:
    ollama = None


# ---------------------------------------------------------------------------
# 1. GUARDRAIL RULES
# ---------------------------------------------------------------------------
# Each rule = (reason_label, compiled_regex)
# Patterns are intentionally broad and case-insensitive to catch common
# prompt-injection / jailbreak phrasing.

INJECTION_PATTERNS = [
    (
        "Attempt to override system instructions",
        re.compile(
            r"\b(ignore|disregard|forget|override)\b.{0,30}\b"
            r"(previous|prior|above|earlier|system|your)\b.{0,30}\b"
            r"(instructions?|guidelines?|rules?|prompt|policy)\b",
            re.IGNORECASE,
        ),
    ),
    (
        "Attempt to extract the system prompt",
        re.compile(
            r"\b(reveal|show|print|leak|display|tell me)\b.{0,30}\b"
            r"(system prompt|hidden prompt|initial prompt|instructions)\b",
            re.IGNORECASE,
        ),
    ),
    (
        "Attempt to change the model's identity/role (jailbreak)",
        re.compile(
            r"\b(you are now|act as|pretend to be|roleplay as|from now on you are)\b"
            r".{0,40}\b(dan|no restrictions|unfiltered|jailbroken|without (any )?rules|"
            r"without (any )?restrictions|evil|unethical)\b",
            re.IGNORECASE,
        ),
    ),
    (
        "Attempt to bypass safety filters",
        re.compile(
            r"\b(bypass|disable|turn off|remove|circumvent)\b.{0,30}\b"
            r"(safety|filters?|guardrails?|restrictions?|content policy|moderation)\b",
            re.IGNORECASE,
        ),
    ),
    (
        "Attempt to gain unrestricted/'do anything now' mode",
        re.compile(
            r"\b(do anything now|dan mode|developer mode|god mode|no ethical guidelines|"
            r"unlimited power|without limitations)\b",
            re.IGNORECASE,
        ),
    ),
    (
        "Request for malicious code or hacking assistance",
        re.compile(
            r"\b(write|create|generate)\b.{0,30}\b"
            r"(malware|ransomware|keylogger|virus|exploit|sql injection|ddos script)\b",
            re.IGNORECASE,
        ),
    ),
    (
        "Request for instructions to build weapons or dangerous substances",
        re.compile(
            r"\b(how to make|how to build|synthesi[sz]e|instructions for)\b.{0,30}\b"
            r"(bomb|explosive|nerve agent|biological weapon|chemical weapon)\b",
            re.IGNORECASE,
        ),
    ),
    (
        "Suspicious delimiter / injection marker used to smuggle new instructions",
        re.compile(
            r"(---\s*end of (user )?prompt\s*---|"
            r"\[/?system\]|<\|.*?\|>|"
            r"^\s*system\s*:\s*|"
            r"###\s*new instructions)",
            re.IGNORECASE,
        ),
    ),
]


def check_prompt(prompt: str):
    """
    Checks the prompt against known injection/unsafe patterns.

    Returns:
        (is_safe: bool, reason: str | None)
    """
    if not prompt or not prompt.strip():
        return False, "Empty prompt."

    for reason, pattern in INJECTION_PATTERNS:
        if pattern.search(prompt):
            return False, reason

    return True, None


# ---------------------------------------------------------------------------
# 2. SEND TO OLLAMA
# ---------------------------------------------------------------------------

def query_ollama(prompt: str, model: str = "llama3") -> str:
    """
    Sends a safe prompt to a locally running Ollama model and returns
    the response text.
    """
    if ollama is None:
        raise RuntimeError(
            "The 'ollama' Python package is not installed. "
            "Run: pip install ollama"
        )

    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )
    return response["message"]["content"]


# ---------------------------------------------------------------------------
# 3. MAIN PROGRAM
# ---------------------------------------------------------------------------

def handle_prompt(prompt: str, model: str = "llama3") -> None:
    print("\n↓ Guardrail")
    is_safe, reason = check_prompt(prompt)

    if is_safe:
        print("Safe")
        print("↓ Sent to Ollama")
        try:
            reply = query_ollama(prompt, model=model)
        except Exception as e:
            print(f"\n[Error contacting Ollama]: {e}")
            return
        print("\nOutput: Response:")
        print(f'"{reply}"')
    else:
        print("Potential Prompt Injection Detected")
        print("Reason:")
        print(f"{reason}")
        print("Request not sent to the LLM.")
        print("\nOutput:\n")
        print("Request Blocked. This prompt violates AI safety guidelines.")


def main():
    model = "llama3"
    if len(sys.argv) > 1:
        model = sys.argv[1]

    print(f"=== Prompt Guardrail for Ollama (model: {model}) ===")
    print("Type your prompt below. Type 'exit' or 'quit' to stop.\n")

    while True:
        try:
            user_prompt = input("User Prompt: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nExiting.")
            break

        if user_prompt.lower() in ("exit", "quit"):
            print("Exiting.")
            break

        handle_prompt(user_prompt, model=model)
        print("\n" + "-" * 60 + "\n")


if __name__ == "__main__":
    main()


=== Prompt Guardrail for Ollama (model: -f) ===
Type your prompt below. Type 'exit' or 'quit' to stop.

